In [1]:
pip install PyQt5


Note: you may need to restart the kernel to use updated packages.


# Personal Expense Tracker

In [ ]:
import sys
import csv
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QFormLayout,
    QLineEdit, QPushButton, QTableWidget, QTableWidgetItem,
    QMessageBox, QLabel, QHBoxLayout
)
from datetime import datetime

FILE_NAME = "expenses.csv"

def init_file():
    try:
        with open(FILE_NAME, 'x', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Date", "Category", "Amount", "Description"])
    except FileExistsError:
        pass

class ExpenseTracker(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Expense Tracker")
        self.setGeometry(100, 100, 700, 500)
        self.setup_ui()
        init_file()
        self.load_expenses()

    def setup_ui(self):
        layout = QVBoxLayout()

        # Add Expense Form
        form = QFormLayout()
        self.date_input = QLineEdit(str(datetime.today().date()))
        self.category_input = QLineEdit()
        self.amount_input = QLineEdit()
        self.desc_input = QLineEdit()
        form.addRow("Date (YYYY-MM-DD):", self.date_input)
        form.addRow("Category:", self.category_input)
        form.addRow("Amount:", self.amount_input)
        form.addRow("Description:", self.desc_input)

        # Action Buttons
        btn_layout = QHBoxLayout()
        add_btn = QPushButton("Add Expense")
        delete_btn = QPushButton("Delete Selected")
        modify_btn = QPushButton("Modify Selected")
        summary_btn = QPushButton("Show Summary")

        add_btn.clicked.connect(self.add_expense)
        delete_btn.clicked.connect(self.delete_selected)
        modify_btn.clicked.connect(self.modify_selected)
        summary_btn.clicked.connect(self.show_summary)

        btn_layout.addWidget(add_btn)
        btn_layout.addWidget(delete_btn)
        btn_layout.addWidget(modify_btn)
        btn_layout.addWidget(summary_btn)

        # Table
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(["Date", "Category", "Amount", "Description"])

        layout.addLayout(form)
        layout.addLayout(btn_layout)
        layout.addWidget(QLabel("All Expenses:"))
        layout.addWidget(self.table)

        self.setLayout(layout)

    def load_expenses(self):
        self.table.setRowCount(0)
        with open(FILE_NAME, 'r') as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row_data in reader:
                row = self.table.rowCount()
                self.table.insertRow(row)
                for col, data in enumerate(row_data):
                    self.table.setItem(row, col, QTableWidgetItem(data))

    def clear_form(self):
        self.category_input.clear()
        self.amount_input.clear()
        self.desc_input.clear()

    def add_expense(self):
        try:
            date = self.date_input.text()
            category = self.category_input.text()
            amount = float(self.amount_input.text())
            desc = self.desc_input.text()
            datetime.strptime(date, "%Y-%m-%d")  # validate

            with open(FILE_NAME, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([date, category, amount, desc])

            QMessageBox.information(self, "Success", "Expense added!")
            self.clear_form()
            self.load_expenses()

        except Exception as e:
            QMessageBox.warning(self, "Error", str(e))

    def delete_selected(self):
        selected = self.table.currentRow()
        if selected < 0:
            QMessageBox.warning(self, "Warning", "No row selected.")
            return

        # Load and remove
        with open(FILE_NAME, 'r') as f:
            rows = list(csv.reader(f))
        del rows[selected + 1]  # offset by header

        with open(FILE_NAME, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(rows)

        QMessageBox.information(self, "Deleted", "Expense removed.")
        self.load_expenses()

    def modify_selected(self):
        selected = self.table.currentRow()
        if selected < 0:
            QMessageBox.warning(self, "Warning", "No row selected.")
            return

        try:
            new_date = self.date_input.text()
            new_cat = self.category_input.text()
            new_amt = float(self.amount_input.text())
            new_desc = self.desc_input.text()
            datetime.strptime(new_date, "%Y-%m-%d")

            # Load data
            with open(FILE_NAME, 'r') as f:
                rows = list(csv.reader(f))

            # Modify selected
            rows[selected + 1] = [new_date, new_cat, str(new_amt), new_desc]

            with open(FILE_NAME, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerows(rows)

            QMessageBox.information(self, "Updated", "Expense updated.")
            self.clear_form()
            self.load_expenses()

        except Exception as e:
            QMessageBox.warning(self, "Error", str(e))

    def show_summary(self):
        total = 0
        categories = {}
        with open(FILE_NAME, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                amount = float(row['Amount'])
                total += amount
                cat = row['Category']
                categories[cat] = categories.get(cat, 0) + amount

        msg = f"Total Spent: ${total:.2f}\n\nBreakdown by Category:\n"
        for cat, amt in categories.items():
            msg += f"  {cat}: ${amt:.2f}\n"

        QMessageBox.information(self, "Summary", msg)

# Run app
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ExpenseTracker()
    window.show()
    sys.exit(app.exec_())

